In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
from scipy.optimize import least_squares

sys.path.insert(0, str(Path("../../nogse_pipeline/src")))
from models.model_fitting import M_ogse_mixed_offset, M_ogse_tort
from plotting.publication.tc_param_vars import _roi_color_map, DEFAULT_BRAIN_MARKERS

In [ ]:
MASTER_PATH  = Path("../../analysis/brains/ogse_experiments/master.long.parquet")
OUT_DIR      = Path("../../analysis/brains/ogse_experiments/lab/fit_mixed-tort")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIRECTIONS  = ["tra", "long"]
N_LIST      = [4, 8]
TD_LIST     = None   # None → all td; list of floats
ROI_LIST    = None
G_COLUMN    = "g_thorsten"
G_CORRECTION_COLUMN = "grad_correction_factor"
Y_COLUMN    = "value_norm"        # "value_norm" or "value"
D0_FIXED    = 3.0e-12   # m²/ms, fixed

TC_FIXED    = "per_curve"  # restricted component tc: None → global; float → pinned; "per_curve" → free
TC_UPPER    = 1000.0

# alpha (restricted/mixed component): None/float/"master"/"per_curve"
# alpha2 (tort component):            None/float/"master"/"per_curve"
ALPHA_FIXED  = "per_curve"
ALPHA2_FIXED = "per_curve"
ALPHA_MASTER_COL  = "alpha_macro"
ALPHA2_MASTER_COL = "alpha_macro"

M0_FIXED    = None      # fraction of mixed component (f1); None → fitted per (td, direction)
C_FIXED     = 0
RN_FIXED    = None      # Rician noise floor (in raw signal units):
                        #   None                 → fitted per td
                        #   float                → fixed for all subj and td
                        #   {td: float}          → fixed per td, same for all subj
                        #   {subj: float}        → fixed per subj, same for all td
                        #   {subj: {td: float}}  → fixed per subj and td

# True  → f2 = 1 − f1  (bimodal fractions constrained to sum to 1)
# False → f2 fitted independently per (td, direction)
CONSTRAIN_FRACTIONS = True

# ── Model name ──────────────────────────────────────────────────────────────
# Model: f1 * M_ogse_mixed(tc, alpha) + f2 * M_ogse_tort(alpha2)  + Rician noise
MODEL_NAME = 'mixed-tort'


In [ ]:
df = pd.read_parquet(MASTER_PATH)

data = df[
    (df.row_kind == "signal_rotated") &
    (df.direction.isin(DIRECTIONS)) &
    (df.N.isin(N_LIST)) &
    (df.stat == "avg")
].copy()

missing_cols = [col for col in [G_COLUMN, Y_COLUMN, G_CORRECTION_COLUMN] if col is not None and col not in data.columns]
if missing_cols:
    raise KeyError(f"Missing selected column(s) in data: {missing_cols}")

data["G_fit"] = data[G_COLUMN]
if G_CORRECTION_COLUMN is not None:
    data["G_fit"] = data["G_fit"] * data[G_CORRECTION_COLUMN]
data["y_raw"] = data["value"]   # raw signal (a.u.) needed to normalise RN per curve
data["y_fit"] = data[Y_COLUMN]

G_LABEL = f"Modulation gradient strength G [mT/m] ({G_COLUMN})"
Y_LABEL = f"{Y_COLUMN} [a.u.]"

if TD_LIST is not None:
    data = data[data.td_ms.isin(TD_LIST)]
if ROI_LIST is not None:
    data = data[data.roi.isin(ROI_LIST)]

print(f"Rows loaded: {len(data)}")
print("Groups (subj, roi, dir):", data.groupby(["subj", "roi", "direction"]).ngroups)
print(f"Gradient for fit: {G_COLUMN}" + (f" * {G_CORRECTION_COLUMN}" if G_CORRECTION_COLUMN else ""))
print(f"Signal for fit: {Y_COLUMN}")
data[["subj", "roi", "direction", "td_ms", "N", "G_fit", "y_fit"]]

In [ ]:
def _get_master_alpha(df_master, subj, roi, direction, col):
    """Look up a fixed alpha value from master for one (subj, roi, dir) group."""
    rows = df_master[
        (df_master.subj == subj) &
        (df_master.roi  == roi) &
        (df_master.direction == direction)
    ]
    if col in rows.columns and not rows[col].isna().all():
        return float(rows[col].dropna().iloc[0])
    raise KeyError(f"Column '{col}' not found or all-NaN for {subj}/{roi}/{direction}")


def _resolve_rn(subj, td):
    """Return RN floor for (subj, td) from RN_FIXED (raw signal units).
    RN_FIXED may be: None → free; float → global;
    {td: float} → per td; {subj: float} → per subj; {subj: {td: float}} → per subj and td.
    """
    if RN_FIXED is None:
        return 0.0
    if isinstance(RN_FIXED, dict):
        first_key = next(iter(RN_FIXED))
        if isinstance(first_key, str):
            subj_entry = RN_FIXED.get(subj)
            if subj_entry is None:
                return 0.0
            if isinstance(subj_entry, dict):
                return float(subj_entry.get(float(td), subj_entry.get(td, 0.0)))
            return float(subj_entry)
        return float(RN_FIXED.get(float(td), RN_FIXED.get(td, 0.0)))
    return float(RN_FIXED)


def fit_group(group_df, alpha_vals, alpha2_vals, subj=None):
    """
    Global fit for one (subj, roi) group across all directions.

    Rician-corrected signal model:
      y_fit = S_meas / M0_norm  =  sqrt((f1·S_mixed + f2·S_tort + C)² + (RN/M0_norm)²)
    where M0_norm = sqrt(S0² − RN²) is the Rician-corrected signal amplitude at G=0.

    CONSTRAIN_FRACTIONS=True  →  f2 = 1 − f1
    CONSTRAIN_FRACTIONS=False →  f2 fitted independently per (td, direction)

    tc     : per-direction or per-curve (TC_FIXED).
    alpha  : per-direction or per-curve (ALPHA_FIXED).
    alpha2 : per-direction or per-curve (ALPHA2_FIXED).
    f1 (M0): mixed-component fraction [0,1], per (td, direction).
    f2     : tort-component fraction [0,1], per (td, direction).
    RN     : Rician noise (raw units), per td.
    C      : offset per (td, N, direction) curve.
    """
    tds = sorted(float(t) for t in group_df.td_ms.unique())
    directions_present = sorted(group_df.direction.unique())

    tc_is_per_curve     = (TC_FIXED == "per_curve")
    alpha_is_per_curve  = {d: (alpha_vals[d]  == "per_curve") for d in directions_present}
    alpha2_is_per_curve = {d: (alpha2_vals[d] == "per_curve") for d in directions_present}

    curves = []
    for direction in directions_present:
        dir_df = group_df[group_df.direction == direction]
        for td in tds:
            for N in N_LIST:
                sub = dir_df[
                    np.isclose(dir_df.td_ms.astype(float), td) & (dir_df.N == N)
                ].sort_values("G_fit")
                if sub.empty:
                    continue
                s0_raw = float(sub.loc[sub.G_fit.abs() == sub.G_fit.abs().min(), "y_raw"].iloc[0])
                curves.append(dict(
                    td=td, N=int(N), direction=direction,
                    G=sub.G_fit.values, y=sub.y_fit.values, S0=s0_raw
                ))

    unique_tds     = sorted(set(c["td"] for c in curves))
    unique_td_dirs = sorted(set((c["td"], c["direction"]) for c in curves))

    def unpack(params):
        idx = 0

        tc_per_dir     = {}
        alpha_per_dir  = {}
        alpha2_per_dir = {}
        for direction in directions_present:
            if TC_FIXED is None:
                tc_per_dir[direction] = np.exp(params[idx]); idx += 1
            elif tc_is_per_curve:
                tc_per_dir[direction] = None
            else:
                tc_per_dir[direction] = float(TC_FIXED)
            if alpha_vals[direction] is None:
                alpha_per_dir[direction] = params[idx]; idx += 1
            elif alpha_is_per_curve[direction]:
                alpha_per_dir[direction] = None
            else:
                alpha_per_dir[direction] = float(alpha_vals[direction])
            if alpha2_vals[direction] is None:
                alpha2_per_dir[direction] = params[idx]; idx += 1
            elif alpha2_is_per_curve[direction]:
                alpha2_per_dir[direction] = None
            else:
                alpha2_per_dir[direction] = float(alpha2_vals[direction])

        per_td = {}
        for td in unique_tds:
            d = {}
            if RN_FIXED is None:
                d["RN"] = float(params[idx]); idx += 1
            else:
                d["RN"] = _resolve_rn(subj, td)
            per_td[td] = d

        per_td_dir = {}
        for (td, direction) in unique_td_dirs:
            d = {}
            if M0_FIXED is None:
                d["M0"] = float(params[idx]); idx += 1
            else:
                d["M0"] = float(M0_FIXED)
            if not CONSTRAIN_FRACTIONS:
                d["f2"] = float(params[idx]); idx += 1
            else:
                d["f2"] = 1.0 - d["M0"]
            per_td_dir[(td, direction)] = d

        per_curve = {}
        for c in curves:
            d = {}
            if tc_is_per_curve:
                d["tc"] = np.exp(params[idx]); idx += 1
            if alpha_is_per_curve[c["direction"]]:
                d["alpha"] = params[idx]; idx += 1
            if alpha2_is_per_curve[c["direction"]]:
                d["alpha2"] = params[idx]; idx += 1
            if C_FIXED is None:
                d["C"] = float(params[idx]); idx += 1
            else:
                d["C"] = float(C_FIXED)
            per_curve[(c["td"], c["N"], c["direction"])] = d

        return tc_per_dir, alpha_per_dir, alpha2_per_dir, per_td, per_td_dir, per_curve

    def residuals(params):
        tc_per_dir, alpha_per_dir, alpha2_per_dir, per_td, per_td_dir, per_curve = unpack(params)
        res = []
        for c in curves:
            p_td_dir = per_td_dir[(c["td"], c["direction"])]
            p        = per_curve[(c["td"], c["N"], c["direction"])]
            rn       = per_td[c["td"]]["RN"]
            tc       = p.get("tc",     tc_per_dir[c["direction"]])
            alpha    = p.get("alpha",  alpha_per_dir[c["direction"]])
            alpha2   = p.get("alpha2", alpha2_per_dir[c["direction"]])
            f1       = p_td_dir["M0"]
            f2       = p_td_dir["f2"]
            M0_norm  = np.sqrt(np.maximum(c["S0"]**2 - rn**2, 1e-20))
            raw_S    = c["y"] * (c["S0"] if Y_COLUMN == "value_norm" else 1.0)
            y_data   = raw_S / M0_norm
            with np.errstate(over="ignore", invalid="ignore"):
                shape_mixed = M_ogse_mixed_offset(c["td"], c["G"], c["N"], c["td"] / c["N"],
                                                  tc, alpha, 1, D0_FIXED, 0, 0)
                shape_tort  = M_ogse_tort(c["td"], c["G"], c["N"], c["td"] / c["N"],
                                          alpha2, 1, D0_FIXED)
                shape = f1 * shape_mixed + f2 * shape_tort + p["C"]
                y_hat = np.sqrt(shape**2 + (rn / M0_norm)**2)
            res.append(y_hat - y_data)
        out = np.concatenate(res)
        return np.where(np.isfinite(out), out, 1e6)

    tc_upper = np.log(TC_UPPER) if TC_UPPER is not None else np.inf
    x0 = []; lower = []; upper = []
    for direction in directions_present:
        if TC_FIXED is None:
            x0 += [np.log(2.0)];  lower += [np.log(0.05)];  upper += [tc_upper]
        if alpha_vals[direction] is None:
            x0 += [0.5];          lower += [0.0];            upper += [1.0]
        if alpha2_vals[direction] is None:
            x0 += [0.5];          lower += [0.0];            upper += [1.0]
    for td in unique_tds:
        if RN_FIXED is None:
            x0 += [35.0];  lower += [0.0];  upper += [500.0]
    for (td, direction) in unique_td_dirs:
        if M0_FIXED is None:
            x0 += [0.5];  lower += [0.0];  upper += [1.0]
        if not CONSTRAIN_FRACTIONS:
            x0 += [0.5];  lower += [0.0];  upper += [1.0]
    for c in curves:
        if tc_is_per_curve:
            x0 += [np.log(2.0)];  lower += [np.log(0.05)];  upper += [tc_upper]
        if alpha_is_per_curve[c["direction"]]:
            x0 += [0.5];          lower += [0.0];            upper += [1.0]
        if alpha2_is_per_curve[c["direction"]]:
            x0 += [0.5];          lower += [0.0];            upper += [1.0]
        if C_FIXED is None:
            x0 += [0.0];          lower += [0.0];            upper += [1.0]

    result = least_squares(residuals, x0, bounds=(lower, upper), method="trf", max_nfev=10000)
    return result, tds, curves, unpack


In [ ]:
rows      = []
fit_store = {}

for (subj, roi), grp in data.groupby(["subj", "roi"]):
    alpha_vals  = {}
    alpha2_vals = {}
    for direction in sorted(grp.direction.unique()):
        if ALPHA_FIXED == "master":
            alpha_vals[direction] = _get_master_alpha(df, subj, roi, direction, ALPHA_MASTER_COL)
        else:
            alpha_vals[direction] = ALPHA_FIXED
        if ALPHA2_FIXED == "master":
            alpha2_vals[direction] = _get_master_alpha(df, subj, roi, direction, ALPHA2_MASTER_COL)
        else:
            alpha2_vals[direction] = ALPHA2_FIXED

    print(f"{subj:6s}  {roi:25s} ...", end="  ")
    result, tds, curves, unpack = fit_group(grp, alpha_vals, alpha2_vals, subj=subj)
    tc_per_dir, alpha_per_dir, alpha2_per_dir, per_td, per_td_dir, per_curve = unpack(result.x)
    directions_fitted = sorted(set(c["direction"] for c in curves))
    for direction in directions_fitted:
        tc_str     = "per_curve" if TC_FIXED     == "per_curve" else f"{tc_per_dir[direction]:.3f} ms"
        alpha_str  = "per_curve" if ALPHA_FIXED  == "per_curve" else f"{alpha_per_dir[direction]:.3f}"
        alpha2_str = "per_curve" if ALPHA2_FIXED == "per_curve" else f"{alpha2_per_dir[direction]:.3f}"
        print(f"{direction}: tc={tc_str}  a={alpha_str}  a2={alpha2_str}", end="  ")
    print(f"cost={result.cost:.3e}")

    fit_store[(subj, roi)] = dict(
        tds=tds, curves=curves, params=result.x,
        directions=directions_fitted,
        tc_per_dir=tc_per_dir, alpha_per_dir=alpha_per_dir, alpha2_per_dir=alpha2_per_dir,
        per_td=per_td, per_td_dir=per_td_dir, per_curve=per_curve
    )

    for c in curves:
        td        = c["td"]
        direction = c["direction"]
        p_td_dir  = per_td_dir[(td, direction)]
        p         = per_curve[(td, c["N"], direction)]
        rows.append(dict(
            subj=subj, roi=roi, direction=direction,
            td_ms=td, N=c["N"],
            tc_ms=p.get("tc",     tc_per_dir[direction]),
            alpha=p.get("alpha",  alpha_per_dir[direction]),
            alpha2=p.get("alpha2", alpha2_per_dir[direction]),
            D0_m2ms=D0_FIXED,
            M0=p_td_dir["M0"], f2=p_td_dir["f2"], C=p["C"], RN=per_td[td]["RN"],
            model_name=MODEL_NAME,
            g_column=G_COLUMN, g_correction_column=G_CORRECTION_COLUMN, y_column=Y_COLUMN,
            cost=float(result.cost), success=bool(result.success),
        ))


In [ ]:
results_df = pd.DataFrame(rows)
results_df.to_excel(OUT_DIR / "fit_results.xlsx", index=False)
print(f"Saved {len(results_df)} rows to {OUT_DIR / 'fit_results.xlsx'}")
results_df

In [ ]:
G_plot = np.linspace(0, float(data.G_fit.max()), 300)
N_hi, N_lo = N_LIST[-1], N_LIST[0]

for (subj, roi), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    for direction in store["directions"]:
        fig, axes = plt.subplots(1, n_tds, figsize=(4 * n_tds, 4.5), sharey=True, squeeze=False)
        axes = axes[0]

        for ax, td in zip(axes, tds):
            title_lines = [f"td = {td:.1f} ms"]
            rn = store["per_td"][td]["RN"]
            y_hat_per_N = {}
            for N, color in zip(N_LIST, ["C0", "C1", "C2", "C3"]):
                c = next((x for x in store["curves"] if x["td"] == td and x["N"] == N and x["direction"] == direction), None)
                if c is None:
                    continue
                p_td_dir = store["per_td_dir"][(td, direction)]
                p        = store["per_curve"][(td, N, direction)]
                tc       = p.get("tc",     store["tc_per_dir"][direction])
                alpha    = p.get("alpha",  store["alpha_per_dir"][direction])
                alpha2   = p.get("alpha2", store["alpha2_per_dir"][direction])
                f1       = p_td_dir["M0"]
                f2       = p_td_dir["f2"]
                M0_norm  = np.sqrt(max(c["S0"]**2 - rn**2, 1e-20))
                raw_S    = c["y"] * (c["S0"] if Y_COLUMN == "value_norm" else 1.0)
                y_data_plot = raw_S / M0_norm
                ax.scatter(c["G"], y_data_plot, color=color, s=20, zorder=3, label=f"N={N} data")
                with np.errstate(over="ignore", invalid="ignore"):
                    shape_mixed = M_ogse_mixed_offset(td, G_plot, N, td / N, tc, alpha, 1, D0_FIXED, 0, 0)
                    shape_tort  = M_ogse_tort(td, G_plot, N, td / N, alpha2, 1, D0_FIXED)
                    shape = f1 * shape_mixed + f2 * shape_tort + p["C"]
                    y_hat = np.sqrt(shape**2 + (rn / M0_norm)**2)
                y_hat_per_N[N] = y_hat
                ax.plot(G_plot, y_hat, color=color, label=f"N={N} fit")
                title_lines.append(f"tc={tc:.2f} ms  a2={alpha2:.3f}  RN={rn:.1f} (N={N})")
            ax.set_title("\n".join(title_lines), fontsize=7)
            ax.set_xlabel(G_LABEL)
            ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)
            if N_hi in y_hat_per_N and N_lo in y_hat_per_N:
                ax_r = ax.twinx()
                ax_r.plot(G_plot, y_hat_per_N[N_hi] - y_hat_per_N[N_lo],
                          color="k", linestyle="--", linewidth=1.2, label=f"ΔS (N={N_hi}−N={N_lo})")
                ax_r.axhline(0, color="gray", linewidth=0.7, linestyle=":")
                ax_r.set_ylabel(f"ΔS (N={N_hi}−N={N_lo})", fontsize=7)
                ax_r.tick_params(axis="y", labelsize=6)

        axes[0].set_ylabel("S / M₀ [a.u.]")
        axes[0].legend(fontsize=6)
        fig.suptitle(f"{subj}  |  {roi}  |  {direction}", fontsize=9)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"fit_{subj}_{roi}_{direction}.png", dpi=120)
        plt.close(fig)

print("Fit plots saved.")


In [ ]:
rois     = sorted(results_df.roi.unique())
subjects = sorted(results_df.subj.unique())
roi_colors = _roi_color_map(rois)
_N_ls        = ["-", "--", "-.", ":"]
N_linestyles = {N: ls for N, ls in zip(N_LIST, _N_ls)}

for direction in DIRECTIONS:
    sub = results_df[results_df.direction == direction]
    for param, ylabel in [("tc_ms", "tc [ms]")]:
        fig, axes = plt.subplots(1, len(rois), figsize=(4 * len(rois), 4), sharey=True)
        axes = np.atleast_1d(axes)

        for ax, roi in zip(axes, rois):
            base_color = roi_colors[roi]
            cmap = mcolors.LinearSegmentedColormap.from_list("", ["#cccccc", base_color])
            for i, subj in enumerate(subjects):
                shade = cmap(0.3 + 0.7 * i / max(1, len(subjects) - 1))
                marker = DEFAULT_BRAIN_MARKERS[i % len(DEFAULT_BRAIN_MARKERS)]
                for N in N_LIST:
                    g = (
                        sub[(sub.roi == roi) & (sub.subj == subj) & (sub.N == N)]
                        .drop_duplicates("td_ms")
                        .sort_values("td_ms")
                    )
                    if g.empty:
                        continue
                    label = subj if N == N_LIST[0] else None
                    ax.scatter(g.td_ms, g[param], color=shade, marker=marker, s=60, zorder=3)
                    ax.plot(g.td_ms, g[param], color=shade, linewidth=0.8,
                            linestyle=N_linestyles[N], label=label)
            ax.set_title(roi, fontsize=9)
            ax.set_xlabel("td [ms]")
            ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)

        axes[0].set_ylabel(ylabel)
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, fontsize=7, loc="upper right", ncol=1)
        n_handles = [
            mlines.Line2D([], [], color="gray", linestyle=N_linestyles[N], label=f"N={N}")
            for N in N_LIST
        ]
        axes[-1].legend(handles=n_handles, fontsize=7, loc="lower right")
        fig.suptitle(f"{param} vs td — direction: {direction}", fontsize=10)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_{direction}.png", dpi=120)
        plt.show()

In [ ]:
for (subj, roi), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    for direction in store["directions"]:
        fig, axes = plt.subplots(1, n_tds, figsize=(4 * n_tds, 4.5), sharey=True, squeeze=False)
        axes = axes[0]

        for ax, td in zip(axes, tds):
            rn = store["per_td"][td]["RN"]
            model_per_N = {}
            for N, color in zip(N_LIST, ["C0", "C1", "C2", "C3"]):
                c = next((x for x in store["curves"] if x["td"] == td and x["N"] == N and x["direction"] == direction), None)
                if c is None:
                    continue
                p_td_dir = store["per_td_dir"][(td, direction)]
                p        = store["per_curve"][(td, N, direction)]
                tc       = p.get("tc",     store["tc_per_dir"][direction])
                alpha    = p.get("alpha",  store["alpha_per_dir"][direction])
                alpha2   = p.get("alpha2", store["alpha2_per_dir"][direction])
                f1       = p_td_dir["M0"]
                f2       = p_td_dir["f2"]
                M0_norm  = np.sqrt(max(c["S0"]**2 - rn**2, 1e-20))
                raw_S    = c["y"] * (c["S0"] if Y_COLUMN == "value_norm" else 1.0)
                y_data_plot = raw_S / M0_norm
                ax.scatter(c["G"], y_data_plot, color=color, s=20, zorder=3, label=f"N={N} data")
                with np.errstate(over="ignore", invalid="ignore"):
                    shape_mixed = M_ogse_mixed_offset(td, G_plot, N, td / N, tc, alpha, 1, D0_FIXED, 0, 0)
                    shape_tort  = M_ogse_tort(td, G_plot, N, td / N, alpha2, 1, D0_FIXED)
                    y_model = f1 * shape_mixed + f2 * shape_tort
                model_per_N[N] = y_model
                ax.plot(G_plot, y_model, color=color, label=f"N={N} model")
            ax.set_title(f"td = {td:.1f} ms", fontsize=8)
            ax.set_xlabel(G_LABEL)
            ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)
            if N_hi in model_per_N and N_lo in model_per_N:
                ax_r = ax.twinx()
                ax_r.plot(G_plot, model_per_N[N_hi] - model_per_N[N_lo],
                          color="k", linestyle="--", linewidth=1.2, label=f"ΔS (N={N_hi}−N={N_lo})")
                ax_r.axhline(0, color="gray", linewidth=0.7, linestyle=":")
                ax_r.set_ylabel(f"ΔS (N={N_hi}−N={N_lo})", fontsize=7)
                ax_r.tick_params(axis="y", labelsize=6)

        axes[0].set_ylabel("S / M₀ [a.u.]")
        axes[0].legend(fontsize=6)
        tc_dir     = store["tc_per_dir"][direction]
        alpha2_dir = store["alpha2_per_dir"][direction]
        tc_label     = "per_curve" if tc_dir     is None else f"{tc_dir:.2f} ms"
        alpha2_label = "per_curve" if alpha2_dir is None else f"{alpha2_dir:.3f}"
        fig.suptitle(
            f"{subj}  |  {roi}  |  {direction}  |  tc={tc_label}  alpha2={alpha2_label}  (clean model)",
            fontsize=9
        )
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"fit_norm_{subj}_{roi}_{direction}.png", dpi=120)
        plt.close(fig)

print("Clean model fit plots saved.")


In [ ]:
subjects_plot = sorted(results_df.subj.unique())
rois_plot     = sorted(results_df.roi.unique())
Ns_plot       = sorted(results_df.N.unique())
dirs_plot     = [d for d in ["long", "tra"] if d in results_df.direction.unique()]
_N_ls_p       = ["-", "--", "-.", ":"]
N_ls          = {N: _N_ls_p[i % len(_N_ls_p)] for i, N in enumerate(Ns_plot)}

for subj in subjects_plot:
    for param, ylabel in [("tc_ms", "tc [ms]"), ("M0", "M0 [a.u.]"), ("C", "C [a.u.]")]:
        fig, axes = plt.subplots(
            len(rois_plot), len(dirs_plot),
            figsize=(4 * len(dirs_plot), 3 * len(rois_plot)),
            sharey=False, squeeze=False
        )
        for i_roi, roi in enumerate(rois_plot):
            for i_dir, direction in enumerate(dirs_plot):
                ax = axes[i_roi, i_dir]
                sub = results_df[
                    (results_df.subj == subj) &
                    (results_df.roi == roi) &
                    (results_df.direction == direction)
                ]
                for N in Ns_plot:
                    g = sub[sub.N == N].sort_values("td_ms")
                    if g.empty:
                        continue
                    line, = ax.plot(g.td_ms, g[param], linestyle=N_ls[N],
                                    marker="o", ms=4, linewidth=0.8, label=f"N={N}")
                    err_col = f"{param}_err"
                    if err_col in g.columns:
                        x = g.td_ms.to_numpy(dtype=float)
                        y = g[param].to_numpy(dtype=float)
                        yerr = g[err_col].to_numpy(dtype=float)
                        ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr)
                        if np.any(ok):
                            ax.fill_between(
                                x[ok], y[ok] - yerr[ok], y[ok] + yerr[ok],
                                color=line.get_color(), alpha=0.18, linewidth=0
                            )
                if i_roi == 0:
                    ax.set_title(direction, fontsize=9)
                if i_dir == 0:
                    ax.set_ylabel(f"{roi}\n{ylabel}", fontsize=8)
                ax.set_xlabel("td [ms]")
                ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                if i_roi == 0 and i_dir == len(dirs_plot) - 1:
                    ax.legend(fontsize=6, title="N", title_fontsize=6)
        fig.suptitle(f"{subj} — {param} vs td", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_{subj}.png", dpi=120)
        plt.close(fig)

In [ ]:
for subj in subjects_plot:
    for param, ylabel in [("tc_ms", "tc [ms]"), ("M0", "M0 [a.u.]"), ("C", "C [a.u.]")]:
        fig, axes = plt.subplots(
            len(rois_plot), len(dirs_plot),
            figsize=(4 * len(dirs_plot), 3 * len(rois_plot)),
            sharey=False, squeeze=False
        )
        for i_roi, roi in enumerate(rois_plot):
            for i_dir, direction in enumerate(dirs_plot):
                ax = axes[i_roi, i_dir]
                sub = results_df[
                    (results_df.subj == subj) &
                    (results_df.roi == roi) &
                    (results_df.direction == direction)
                ]
                for N in Ns_plot:
                    g = sub[sub.N == N].sort_values("td_ms")
                    if g.empty:
                        continue
                    line, = ax.plot(g.td_ms, g[param], linestyle=N_ls[N],
                                    marker="o", ms=4, linewidth=0.8, label=f"N={N}")
                    err_col = f"{param}_err"
                    if err_col in g.columns:
                        x = g.td_ms.to_numpy(dtype=float)
                        y = g[param].to_numpy(dtype=float)
                        yerr = g[err_col].to_numpy(dtype=float)
                        lower = np.maximum(y - yerr, np.finfo(float).tiny)
                        upper = y + yerr
                        ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr) & (y > 0) & (upper > 0)
                        if np.any(ok):
                            ax.fill_between(
                                x[ok], lower[ok], upper[ok],
                                color=line.get_color(), alpha=0.18, linewidth=0
                            )
                ax.set_yscale("log")
                if i_roi == 0:
                    ax.set_title(direction, fontsize=9)
                if i_dir == 0:
                    ax.set_ylabel(f"{roi}\n{ylabel} (log)", fontsize=8)
                ax.set_xlabel("td [ms]")
                ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                if i_roi == 0 and i_dir == len(dirs_plot) - 1:
                    ax.legend(fontsize=6, title="N", title_fontsize=6)
        fig.suptitle(f"{subj} — {param} vs td  (log scale)", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_log_{subj}.png", dpi=120)
        plt.close(fig)

In [ ]:
# ── Per-subject contrast figures: ROIs × directions, contrast curves per td ──
# _raw : contrast from full fit  sqrt((f1·S1+f2·S2+C)² + (RN/M₀)²)  for N=8 minus N=4
# _corr: contrast from clean model  f1·S1 + f2·S2  for N=8 minus N=4
td_cmap    = plt.cm.viridis
N_hi, N_lo = 8, 4
G_plot     = np.linspace(0, float(data.G_fit.max()), 300)
PEAK_GAMMA = 267.5221900   # rad / (ms · mT), proton gyromagnetic ratio

def _x_for_td(td_ms, xvar):
    """Return x-axis array for the given variable, derived from G_plot."""
    if xvar == "G":
        return G_plot.copy()
    D0, gamma = float(D0_PLOT), float(PEAK_GAMMA)
    l_d = np.sqrt(D0 * float(td_ms))
    l_G = np.full_like(G_plot, np.nan)
    valid = G_plot > 0
    l_G[valid] = (D0 / (gamma * G_plot[valid])) ** (1.0 / 3.0)
    Ld  = l_d / l_G
    Lcf = 1.5 ** 0.25 / np.sqrt(Ld)
    lcf = Lcf * l_G * 1e6  # µm
    if xvar == "Ld":  return Ld
    if xvar == "Lcf": return Lcf
    if xvar == "lcf": return lcf
    raise ValueError(xvar)

X_AXES = [
    ("G",   G_LABEL),
    ("Ld",  "Lᵈ (dimensionless)"),
    ("Lcf", "Lcf (dimensionless)"),
    ("lcf", "lcf [µm]"),
]

D0_PLOT = D0_FIXED

# ── precompute both contrast stores ──────────────────────────────────────────
raw_contrast_store  = {}   # full fit with Rician noise floor
corr_contrast_store = {}   # clean model f1·S1 + f2·S2

for (subj, roi), store in fit_store.items():
    for direction in store["directions"]:
        for td in store["tds"]:
            rn = store["per_td"][td]["RN"]
            raw_per_N  = {}
            corr_per_N = {}
            for N in N_LIST:
                c = next((x for x in store["curves"]
                          if x["td"] == td and x["N"] == N and x["direction"] == direction), None)
                if c is None:
                    continue
                p_td_dir = store["per_td_dir"][(td, direction)]
                p        = store["per_curve"][(td, N, direction)]
                tc       = p.get("tc",     store["tc_per_dir"][direction])
                alpha    = p.get("alpha",  store["alpha_per_dir"][direction])
                alpha2   = p.get("alpha2", store["alpha2_per_dir"][direction])
                f1       = p_td_dir["M0"]
                f2       = p_td_dir["f2"]
                M0_norm  = np.sqrt(np.maximum(c["S0"]**2 - rn**2, 1e-20))
                with np.errstate(over="ignore", invalid="ignore"):
                    shape_m = M_ogse_mixed_offset(td, G_plot, N, td / N, tc, alpha, 1, D0_FIXED, 0, 0)
                    shape_t = M_ogse_tort(td, G_plot, N, td / N, alpha2, 1, D0_FIXED)
                    y_model = f1 * shape_m + f2 * shape_t
                    y_raw   = np.sqrt((y_model + p["C"])**2 + (rn / M0_norm)**2)
                raw_per_N[N]  = y_raw
                corr_per_N[N] = y_model
            if N_hi in raw_per_N and N_lo in raw_per_N:
                raw_contrast_store[(subj, roi, direction, td)]  = raw_per_N[N_hi]  - raw_per_N[N_lo]
                corr_contrast_store[(subj, roi, direction, td)] = corr_per_N[N_hi] - corr_per_N[N_lo]

# ── shared plotting helper ────────────────────────────────────────────────────
def _plot_contrast(cstore, xvar, xlabel, subj, suffix, title_tag):
    rois_subj = sorted(set(k[1] for k in cstore if k[0] == subj))
    dirs_subj = [d for d in DIRECTIONS if any(k[2] == d for k in cstore if k[0] == subj)]
    n_rois, n_dirs = len(rois_subj), len(dirs_subj)
    if n_rois == 0 or n_dirs == 0:
        return

    all_tds_subj = sorted(set(k[3] for k in cstore if k[0] == subj))
    colors_leg   = [td_cmap(i / max(1, len(all_tds_subj) - 1)) for i in range(len(all_tds_subj))]
    leg_handles  = [plt.Line2D([0], [0], color=c, linewidth=1.5) for c in colors_leg]
    leg_labels   = [f"td={td:.0f} ms" for td in all_tds_subj]
    td_color_map = {td: c for td, c in zip(all_tds_subj, colors_leg)}

    fig, axes = plt.subplots(n_rois, n_dirs, figsize=(4 * n_dirs, 3 * n_rois), squeeze=False)
    for i_roi, roi in enumerate(rois_subj):
        for i_dir, direction in enumerate(dirs_subj):
            ax = axes[i_roi, i_dir]
            tds_here = sorted(k[3] for k in cstore
                               if k[0] == subj and k[1] == roi and k[2] == direction)
            if not tds_here:
                ax.set_visible(False)
                continue
            for td in tds_here:
                ax.plot(_x_for_td(td, xvar), cstore[(subj, roi, direction, td)],
                        color=td_color_map[td], linewidth=1.2)
            ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
            if i_roi == 0:
                ax.set_title(direction, fontsize=9)
            if i_dir == 0:
                ax.set_ylabel(f"{roi}\nΔS (N={N_hi}−N={N_lo})", fontsize=7)
            ax.set_xlabel(xlabel, fontsize=7)
            ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)
            ax.tick_params(labelsize=6)

    fig.suptitle(f"{subj} — ΔS (N={N_hi}−N={N_lo}) vs {xvar}  [{title_tag}]", fontsize=10)
    fig.legend(leg_handles, leg_labels,
               loc="upper center", ncol=len(leg_labels), fontsize=7, frameon=False,
               bbox_to_anchor=(0.5, 1.04), bbox_transform=fig.transFigure)
    fig.tight_layout(rect=[0, 0, 1, 0.87])
    fig.savefig(OUT_DIR / f"contrast_vs_{xvar}_{subj}_{suffix}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

# ── generate figures ──────────────────────────────────────────────────────────
all_subjs = sorted(set(k[0] for k in raw_contrast_store))
for xvar, xlabel in X_AXES:
    for subj in all_subjs:
        _plot_contrast(raw_contrast_store,  xvar, xlabel, subj, "raw",  "raw fit w/ RN")
        _plot_contrast(corr_contrast_store, xvar, xlabel, subj, "corr", "clean model")

print("Per-subject contrast figures saved.")
